In [1]:
import xarray as xr
ruta_precip = '/home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS/vertically_integrated_moisture_divergence.nc'
ds = xr.open_dataset(ruta_precip)
print("=== DIMENSIONES ===")
print(ds.dims)
print("\n=== VARIABLES ===")
print(list(ds.data_vars))
print("\n=== COORDENADAS ===")
print(ds.coords)
print("\n=== VISTA GENERAL ===")
print(ds)

=== DIMENSIONES ===
FrozenMappingWarningOnValuesAccess({'valid_time': 828, 'latitude': 287, 'longitude': 207})

=== VARIABLES ===
['avg_vimdf']

=== COORDENADAS ===
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 7kB 1950-01-01T06:00:00 ... 2018-...
    expver      (valid_time) <U4 13kB ...
  * latitude    (latitude) float64 2kB 15.0 14.75 14.5 ... -56.0 -56.25 -56.5
  * longitude   (longitude) float64 2kB -84.0 -83.75 -83.5 ... -32.75 -32.5
    number      int64 8B ...

=== VISTA GENERAL ===
<xarray.Dataset> Size: 197MB
Dimensions:     (valid_time: 828, latitude: 287, longitude: 207)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 7kB 1950-01-01T06:00:00 ... 2018-...
    expver      (valid_time) <U4 13kB ...
  * latitude    (latitude) float64 2kB 15.0 14.75 14.5 ... -56.0 -56.25 -56.5
  * longitude   (longitude) float64 2kB -84.0 -83.75 -83.5 ... -32.75 -32.5
    number      int64 8B ...
Data variables:
    avg_vimdf   (valid_time, latitude, longitude) float32 197MB 

In [1]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import psutil
import gc
from tqdm import tqdm
#area": [15, -84, -56.5, -32.5]
# ============================================================================
# 1. Rutas (cámbialas si es necesario)
# ============================================================================
ruta_precip = '/home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS/vertically_integrated_moisture_divergence.nc'
ruta_nino   = '/home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS/datos_nino.xlsx'
carpeta_salida = '/home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS'
os.makedirs(carpeta_salida, exist_ok=True)

# ============================================================================
# Función para monitorear memoria
# ============================================================================
def print_memory_usage(stage):
    mem = psutil.Process().memory_info().rss / 1024 / 1024
    print(f"[MEMORIA] {stage}: {mem:.1f} MB")

# ============================================================================
# 2. Cargar archivo, renombrar dimensiones y seleccionar variable
# ============================================================================
print("Cargando archivo NetCDF...")
ds = xr.open_dataset(ruta_precip)

# Renombrar dimensiones a 'time', 'lat', 'lon'
ds = ds.rename({
    'valid_time': 'time',
    'latitude': 'lat',
    'longitude': 'lon'
})

# Seleccionar la variable (divergencia de humedad)
var = ds['avg_vimdf']   # unidades originales: kg/m²/s (o similar)
print_memory_usage("Después de cargar variable original")

# ============================================================================
# 3. Reducción espacial (coarsening) - opcional, por defecto factor=1
# ============================================================================
coarsen_factor = 1   # 1 = sin reducción; prueba 2 si memoria es problema
if coarsen_factor > 1:
    var = var.coarsen(lat=coarsen_factor, lon=coarsen_factor, boundary='trim').mean()
print(f"Nueva dimensión espacial: {var.lat.size} lat × {var.lon.size} lon")
print_memory_usage("Después de coarsen")

# ============================================================================
# 4. Ajustar tiempo al inicio del mes y recortar a 1950-2018
# ============================================================================
# Las horas en 'valid_time' son 06:00:00; las llevamos al día (sin hora)
var['time'] = var.time.dt.floor('D')
fecha_inicio = '1951-01-01'
fecha_fin    = '2018-12-31'
var = var.sel(time=slice(fecha_inicio, fecha_fin))
print(f"Rango temporal variable: {var.time.min().values} → {var.time.max().values}")
print_memory_usage("Después de recorte temporal 1950-2018")

# ============================================================================
# 5. Anomalía estandarizada mensual
# ============================================================================
print("Calculando anomalía estandarizada mensual (1950-2018)...")
monthly_mean = var.groupby('time.month').mean('time')
monthly_std  = var.groupby('time.month').std('time')

def standardize_by_month(da, mean_da, std_da):
    meses = da['time'].dt.month
    std_anom = (da - mean_da.sel(month=meses)) / std_da.sel(month=meses)
    return std_anom

var_anom_std = standardize_by_month(var, monthly_mean, monthly_std)
print_memory_usage("Después de anomalía estandarizada")

# ============================================================================
# 6. Cargar índice ENSO 3.4 y recortar al mismo período
# ============================================================================
print("Cargando ENSO...")
df = pd.read_excel(ruta_nino)
df_fechas = df[['YR', 'MON']].rename(columns={'YR': 'year', 'MON': 'month'})
df_fechas['day'] = 1
df['date'] = pd.to_datetime(df_fechas)
enso = df.set_index('date')['ANOM']
enso.index = enso.index.rename('time')
enso_da = xr.DataArray(enso, dims=['time'])
enso_da = enso_da.sel(time=slice(fecha_inicio, fecha_fin))
print(f"Rango temporal ENSO: {enso_da.time.min().values} → {enso_da.time.max().values}")
print_memory_usage("Después de cargar ENSO")

# ============================================================================
# 7. Alinear series por intersección de fechas
# ============================================================================
time_var = var_anom_std.time.values
time_enso = enso_da.time.values
time_comun = np.intersect1d(time_var, time_enso)
time_comun = pd.DatetimeIndex(time_comun)
if len(time_comun) != len(time_var) or len(time_comun) != len(time_enso):
    print("Advertencia: fechas no coinciden. Usando intersección.")
    var_anom_std = var_anom_std.sel(time=time_comun)
    enso_da = enso_da.sel(time=time_comun)
else:
    print("Fechas coinciden exactamente.")
print(f"Período común: {time_comun.min()} → {time_comun.max()}")
print_memory_usage("Después de alinear")

# ============================================================================
# 8. Correlación celda por celda
# ============================================================================
print("Calculando correlación celda a celda...")
latitudes = var_anom_std.lat.values
longitudes = var_anom_std.lon.values
corr_matrix = np.full((len(latitudes), len(longitudes)), np.nan, dtype=np.float32)
enso_values = enso_da.values

for i, lat in enumerate(tqdm(latitudes, desc="Latitudes")):
    for j, lon in enumerate(longitudes):
        serie = var_anom_std.isel(lat=i, lon=j).values
        mask = ~np.isnan(serie)
        if np.sum(mask) > 10:
            corr_matrix[i, j] = np.corrcoef(serie[mask], enso_values[mask])[0, 1]
    if i % 20 == 0:
        gc.collect()

max_corr = np.nanmax(corr_matrix)
min_corr = np.nanmin(corr_matrix)
print(f"Correlación: mínimo = {min_corr:.4f}, máximo = {max_corr:.4f}")
print(f"Rango completo: {min_corr:.3f} a {max_corr:.3f}")
print_memory_usage("Después de correlación")

# ============================================================================
# 9. Guardar resultado en NetCDF (CF compliant para GrADS)
# ============================================================================
print("Preparando y guardando resultado...")
corr_da = xr.DataArray(
    corr_matrix,
    dims=('lat', 'lon'),
    coords={'lat': latitudes, 'lon': longitudes},
    name='correlacion',
    attrs={
        'long_name': 'Correlación de Pearson (anomalía estandarizada de avg_vimdf vs ENSO 3.4)',
        'units': 'adimensional'
    }
)
# Ordenar coordenadas (lat creciente, lon creciente)
corr_da = corr_da.sortby('lat')
corr_da = corr_da.sortby('lon')
corr_da.lat.attrs = {'standard_name': 'latitude', 'units': 'degrees_north'}
corr_da.lon.attrs = {'standard_name': 'longitude', 'units': 'degrees_east'}

ds_out = xr.Dataset({'correlacion': corr_da})
ds_out.attrs['title'] = 'Correlación ENSO - Divergencia de humedad integrada'
ds_out.attrs['source_var'] = 'ERA5 avg_vimdf'
ds_out.attrs['periodo'] = f'{fecha_inicio} a {fecha_fin}'
ds_out.attrs['metodo'] = 'Anomalía mensual estandarizada (media y std por mes)'

# Encontrar el índice del valor máximo y mínimo (ignorando NaNs)
max_idx = np.nanargmax(corr_matrix)
min_idx = np.nanargmin(corr_matrix)
max_pos = np.unravel_index(max_idx, corr_matrix.shape)  # (i, j)
min_pos = np.unravel_index(min_idx, corr_matrix.shape)
lat_max = latitudes[max_pos[0]]
lon_max = longitudes[max_pos[1]]
lat_min = latitudes[min_pos[0]]
lon_min = longitudes[min_pos[1]]
max_val = np.nanmax(corr_matrix)
min_val = np.nanmin(corr_matrix)

print(f"Correlación máxima: {max_val:.4f} en (lat={lat_max:.3f}°, lon={lon_max:.3f}°)")
print(f"Correlación mínima: {min_val:.4f} en (lat={lat_min:.3f}°, lon={lon_min:.3f}°)")
# Valor más cercano a cero (en valor absoluto)
abs_corr = np.abs(corr_matrix)
flat_abs = abs_corr.flatten()
# Ignorar NaNs
mask_finite = ~np.isnan(flat_abs)
flat_abs_finite = flat_abs[mask_finite]
min_abs_val = np.min(flat_abs_finite)
# Índice del mínimo absoluto en la matriz aplanada
idx_zero_abs = np.nanargmin(abs_corr)   # funciona con numpy 1.17+
pos_zero = np.unravel_index(idx_zero_abs, corr_matrix.shape)
lat_zero = latitudes[pos_zero[0]]
lon_zero = longitudes[pos_zero[1]]
# El valor real (con signo)
zero_val = corr_matrix[pos_zero]

print(f"\n>>> Correlación más cercana a cero: {zero_val:.6f} (abs={min_abs_val:.6f})")
print(f"    en (lat={lat_zero:.3f}°, lon={lon_zero:.3f}°)")

ruta_nc = os.path.join(carpeta_salida, 'correlacion_SOI_vimdf_cf.nc')
ds_out.to_netcdf(ruta_nc, mode='w')
print(f"Correlación guardada en: {ruta_nc}")
print_memory_usage("Final")

Cargando archivo NetCDF...
[MEMORIA] Después de cargar variable original: 201.4 MB
Nueva dimensión espacial: 287 lat × 207 lon
[MEMORIA] Después de coarsen: 201.4 MB
Rango temporal variable: 1951-01-01T00:00:00.000000000 → 2018-12-01T00:00:00.000000000
[MEMORIA] Después de recorte temporal 1950-2018: 202.8 MB
Calculando anomalía estandarizada mensual (1950-2018)...
[MEMORIA] Después de anomalía estandarizada: 494.2 MB
Cargando ENSO...
Rango temporal ENSO: 1951-01-01T00:00:00.000000 → 2018-12-01T00:00:00.000000
[MEMORIA] Después de cargar ENSO: 500.7 MB
Fechas coinciden exactamente.
Período común: 1951-01-01 00:00:00 → 2018-12-01 00:00:00
[MEMORIA] Después de alinear: 500.7 MB
Calculando correlación celda a celda...


Latitudes: 100%|██████████| 287/287 [00:27<00:00, 10.43it/s]


Correlación: mínimo = -0.6507, máximo = 0.5993
Rango completo: -0.651 a 0.599
[MEMORIA] Después de correlación: 500.9 MB
Preparando y guardando resultado...
Correlación máxima: 0.5993 en (lat=-14.250°, lon=-77.000°)
Correlación mínima: -0.6507 en (lat=-2.750°, lon=-79.750°)

>>> Correlación más cercana a cero: -0.000003 (abs=0.000003)
    en (lat=14.750°, lon=-45.750°)
Correlación guardada en: /home/santiago/Escritorio/UNIVERSIDAD/FISICA_DEL_CLIMA/F-sica-del-Clima/proyecto_4/DATOS/correlacion_SOI_vimdf_cf.nc
[MEMORIA] Final: 501.1 MB
